# Evaluate CheXpert E2 on Kaggle valid.csv

Notebook mới dựa trên cấu hình E2 cũ, có chuẩn hóa đường dẫn ảnh trong thư mục valid.

In [ ]:
from pathlib import Path
import json, sys, importlib.util
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

INPUT_ROOT = Path('/kaggle/input')
VALID_CSV = INPUT_ROOT / 'datasets/ashery/chexpert/valid.csv'
CHECKPOINT_PATH = INPUT_ROOT / 'models/thanhdaydungso/densenet-e2/pytorch/default/1/best_E2_relation_aware_weighted_bce.pt'
if not VALID_CSV.is_file():
    candidates = sorted(INPUT_ROOT.rglob('valid.csv'))
    assert candidates, 'Không tìm thấy valid.csv'
    VALID_CSV = next((p for p in candidates if 'chexpert' in str(p).lower()), candidates[0])
if not CHECKPOINT_PATH.is_file():
    candidates = sorted(INPUT_ROOT.rglob('best_E2_relation_aware_weighted_bce.pt'))
    assert candidates, 'Không tìm thấy checkpoint'
    CHECKPOINT_PATH = candidates[0]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAIN_LABELS = ['Atelectasis','Cardiomegaly','Edema','Pleural Effusion','Pneumonia','Pneumothorax']
print('VALID_CSV:', VALID_CSV); print('CHECKPOINT:', CHECKPOINT_PATH); print('DEVICE:', DEVICE)

In [ ]:
df = pd.read_csv(VALID_CSV)
df = df[df['Frontal/Lateral'].astype(str).str.lower().eq('frontal')].reset_index(drop=True)
TRANSFORM = transforms.Compose([transforms.Resize((320,320)), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

class ValidDataset(Dataset):
    def __init__(self, frame): self.frame, self.file_index = frame, None
    def resolve_path(self, value):
        raw = str(value).replace('\\','/').lstrip('/')
        prefix = 'CheXpert-v1.0-small/'
        if raw.lower().startswith(prefix.lower()): raw = raw[len(prefix):]
        parts = raw.split('/')
        relative = '/'.join(parts[1:]) if parts and parts[0].lower() == 'valid' else raw
        roots = [VALID_CSV.parent, VALID_CSV.parent.parent, VALID_CSV.parent.parent.parent, INPUT_ROOT]
        for root in roots:
            for candidate in (root / raw, root / relative, root / 'valid' / relative):
                if candidate.is_file(): return candidate
        if self.file_index is None:
            self.file_index = {}
            for root in roots:
                valid = root if root.name.lower() == 'valid' else root / 'valid'
                if valid.is_dir():
                    for candidate in valid.rglob('*'):
                        if candidate.is_file() and candidate.suffix.lower() in {'.jpg','.jpeg','.png'}: self.file_index[candidate.relative_to(valid).as_posix().lower()] = candidate
        if relative.lower() in self.file_index: return self.file_index[relative.lower()]
        raise FileNotFoundError(f'Không tìm thấy ảnh sau chuẩn hóa: valid/{relative}')
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        with Image.open(self.resolve_path(self.frame.iloc[index]['Path'])) as source: image = source.convert('RGB')
        return TRANSFORM(image), index

loader = DataLoader(ValidDataset(df), batch_size=32, shuffle=False, num_workers=0)
print('Frontal validation images:', len(df), 'batches:', len(loader))

## Load model

Copy phần `RelationAwareDenseNet121`, `positive_probability` và forward inference từ `scripts/predict_chexpert_6labels.py` vào cell kế tiếp, hoặc thêm repository vào Kaggle Dataset rồi import script đó.

In [ ]:
# Nếu đã upload repository dưới /kaggle/input/image-rag, dùng đoạn này để import model.
SCRIPT_PATH = next(iter(Path('/kaggle/input').rglob('predict_chexpert_6labels.py')), None)
assert SCRIPT_PATH is not None, 'Hãy upload scripts/predict_chexpert_6labels.py cùng notebook.'
spec = importlib.util.spec_from_file_location('predict_model', SCRIPT_PATH)
predict_model = importlib.util.module_from_spec(spec); sys.modules[spec.name] = predict_model; spec.loader.exec_module(predict_model)
model, checkpoint = predict_model.load_model(CHECKPOINT_PATH, DEVICE)
probabilities = np.zeros((len(df), len(MAIN_LABELS)), dtype=np.float32)
with torch.inference_mode():
    for images, indices in loader:
        outputs = model(images.to(DEVICE))
        values = torch.stack([predict_model.positive_probability_from_logits(label, outputs[label], model.policies) for label in MAIN_LABELS], dim=1)
        probabilities[indices.numpy()] = values.cpu().numpy()
print('Predictions:', probabilities.shape)

In [ ]:
metric_rows = []
for index, label in enumerate(MAIN_LABELS):
    values = pd.to_numeric(df[label], errors='coerce').to_numpy()
    valid = np.isin(values, [0., 1.]); y = values[valid].astype(int); p = probabilities[valid, index]; pred = (p >= .5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    metric_rows.append({'Label':label,'N':len(y),'Positive':int(y.sum()),'AUROC':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,'AUPRC':average_precision_score(y,p) if y.sum() else np.nan,'Accuracy':accuracy_score(y,pred),'Precision':precision_score(y,pred,zero_division=0),'Recall':recall_score(y,pred,zero_division=0),'F1':f1_score(y,pred,zero_division=0),'Specificity':tn/(tn+fp) if tn+fp else np.nan})
metrics = pd.DataFrame(metric_rows); display(metrics.round(4)); print('Macro AUROC:', metrics.AUROC.mean()); print('Macro AUPRC:', metrics.AUPRC.mean())
out = Path('/kaggle/working/chexpert_e2_valid_results'); out.mkdir(exist_ok=True); metrics.to_csv(out/'metrics.csv', index=False); pd.DataFrame(probabilities, columns=MAIN_LABELS).assign(Path=df.Path).to_csv(out/'predictions.csv', index=False)